# Exploratory Analysis and Hypothesis Validation

## Objectives

This notebook:

- creates complete-year global and country summaries;
- calculates temperature anomalies using a documented baseline;
- explores long-term temperature and uncertainty trends;
- validates the two project hypotheses;
- communicates statistical results in accessible language;
- creates smaller analytical datasets for the Streamlit dashboard.

Annual averages are used for hypothesis testing to reduce the risk of treating monthly observations as fully independent measurements.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats


PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "jupyter_notebooks"
    else Path.cwd()
)

PROCESSED_FOLDER = PROJECT_ROOT / "data" / "processed" / "v1"

GLOBAL_FILE = (
    PROCESSED_FOLDER / "global_temperatures_clean.csv"
)

COUNTRY_FILE = (
    PROCESSED_FOLDER / "country_temperatures_clean.csv"
)

global_clean = pd.read_csv(
    GLOBAL_FILE,
    parse_dates=["date"],
)

country_clean = pd.read_csv(
    COUNTRY_FILE,
    parse_dates=["date"],
)

print(f"Global cleaned shape: {global_clean.shape}")
print(f"Country cleaned shape: {country_clean.shape}")

Global cleaned shape: (3180, 11)
Country cleaned shape: (544811, 6)


## Global Annual Summary

Monthly measurements are aggregated into annual averages. Observation counts are retained so incomplete years can be identified.

The 1951–1980 mean is used as the project's historical anomaly baseline.
An anomaly indicates how much warmer or cooler a year was than this baseline.

In [2]:
global_annual = (
    global_clean
    .groupby("year", as_index=False)
    .agg(
        land_average_temperature_c=(
            "land_average_temperature_c",
            "mean",
        ),
        land_average_uncertainty_c=(
            "land_average_temperature_uncertainty_c",
            "mean",
        ),
        land_ocean_average_temperature_c=(
            "land_ocean_average_temperature_c",
            "mean",
        ),
        land_ocean_average_uncertainty_c=(
            "land_ocean_average_temperature_uncertainty_c",
            "mean",
        ),
        land_months=(
            "land_average_temperature_c",
            "count",
        ),
        land_ocean_months=(
            "land_ocean_average_temperature_c",
            "count",
        ),
    )
    .sort_values("year")
    .reset_index(drop=True)
)

# Calculate the 1951–1980 land-and-ocean baseline.
baseline_mask = (
    global_annual["year"].between(1951, 1980)
    & global_annual["land_ocean_months"].eq(12)
)

global_baseline_c = global_annual.loc[
    baseline_mask,
    "land_ocean_average_temperature_c",
].mean()

global_annual["land_ocean_anomaly_c"] = (
    global_annual["land_ocean_average_temperature_c"]
    - global_baseline_c
)

print(f"Global baseline: {global_baseline_c:.3f} °C")
print(f"Annual summary shape: {global_annual.shape}")

display(global_annual.tail())

Global baseline: 15.300 °C
Annual summary shape: (266, 8)


,year,land_average_temperature_c,land_average_uncertainty_c,land_ocean_average_temperature_c,land_ocean_average_uncertainty_c,land_months,land_ocean_months,land_ocean_anomaly_c
261,2011,9.516000,0.082000,15.769500,0.059000,12,12,0.469953
262,2012,9.507333,0.083417,15.802333,0.061500,12,12,0.502786
263,2013,9.606500,0.097667,15.854417,0.064667,12,12,0.554869
264,2014,9.570667,0.090167,15.913000,0.063167,12,12,0.613453
265,2015,9.831000,0.092167,16.058583,0.060833,12,12,0.759036


In [3]:
import sys
import nbformat

global_ocean_annual = global_annual.loc[
    global_annual["land_ocean_months"].eq(12)
].copy()

global_ocean_annual["anomaly_10_year_mean_c"] = (
    global_ocean_annual["land_ocean_anomaly_c"]
    .rolling(window=10, min_periods=10)
    .mean()
)

figure = go.Figure()

figure.add_trace(
    go.Scatter(
        x=global_ocean_annual["year"],
        y=global_ocean_annual["land_ocean_anomaly_c"],
        name="Annual anomaly",
        mode="lines",
        line={
            "color": "#4C78A8",
            "width": 1.5,
        },
        hovertemplate=(
            "Year: %{x}<br>"
            "Anomaly: %{y:.2f} °C"
            "<extra></extra>"
        ),
    )
)

figure.add_trace(
    go.Scatter(
        x=global_ocean_annual["year"],
        y=global_ocean_annual["anomaly_10_year_mean_c"],
        name="10-year rolling mean",
        mode="lines",
        line={
            "color": "#E45756",
            "width": 3,
        },
        hovertemplate=(
            "Year: %{x}<br>"
            "Rolling anomaly: %{y:.2f} °C"
            "<extra></extra>"
        ),
    )
)

figure.add_hline(
    y=0,
    line_dash="dash",
    line_color="#555555",
)

figure.update_layout(
    title=(
        "Global Land-and-Ocean Temperature Anomaly "
        "Relative to 1951–1980"
    ),
    xaxis_title="Year",
    yaxis_title="Temperature anomaly (°C)",
    template="plotly_white",
    legend_title="Measurement",
    hovermode="x unified",
)

figure.show()

### Global Trend Interpretation

The annual anomaly fluctuates from year to year, but the rolling average shows a clear long-term increase. Recent decades are consistently warmer than the
1951–1980 project baseline.

An anomaly is more useful than absolute temperature for communicating change because it focuses on differences from a consistent reference period.

## Hypothesis 1

> The mean global land-and-ocean temperature during 1986–2015 is higher than during 1956–1985.

### Method

- Monthly data is aggregated into complete annual averages.
- Both comparison periods contain 30 years.
- A two-sided Welch t-test is used because equal variance is not assumed.
- Cohen's d is calculated to describe the size of the difference.

In [4]:
def cohens_d(group_a, group_b):
    """Calculate Cohen's d using the pooled sample standard deviation."""
    group_a = pd.Series(group_a).dropna()
    group_b = pd.Series(group_b).dropna()

    pooled_variance = (
        (
            (len(group_a) - 1) * group_a.var(ddof=1)
            + (len(group_b) - 1) * group_b.var(ddof=1)
        )
        / (len(group_a) + len(group_b) - 2)
    )

    return (
        group_a.mean() - group_b.mean()
    ) / np.sqrt(pooled_variance)

In [5]:
h1_earlier = global_ocean_annual.loc[
    global_ocean_annual["year"].between(1956, 1985),
    "land_ocean_average_temperature_c",
]

h1_later = global_ocean_annual.loc[
    global_ocean_annual["year"].between(1986, 2015),
    "land_ocean_average_temperature_c",
]

h1_test = stats.ttest_ind(
    h1_later,
    h1_earlier,
    equal_var=False,
)

h1_difference_c = h1_later.mean() - h1_earlier.mean()
h1_effect_size = cohens_d(h1_later, h1_earlier)

print(f"1956–1985 observations: {len(h1_earlier)}")
print(f"1986–2015 observations: {len(h1_later)}")
print(f"1956–1985 mean: {h1_earlier.mean():.4f} °C")
print(f"1986–2015 mean: {h1_later.mean():.4f} °C")
print(f"Difference: {h1_difference_c:.4f} °C")
print(f"Welch t-statistic: {h1_test.statistic:.4f}")
print(f"Two-sided p-value: {h1_test.pvalue:.3e}")
print(f"Cohen's d: {h1_effect_size:.3f}")

1956–1985 observations: 30
1986–2015 observations: 30
1956–1985 mean: 15.3183 °C
1986–2015 mean: 15.7014 °C
Difference: 0.3831 °C
Welch t-statistic: 10.5009
Two-sided p-value: 3.207e-14
Cohen's d: 2.711


In [6]:
h1_plot_data = pd.concat(
    [
        pd.DataFrame({
            "period": "1956–1985",
            "annual_temperature_c": h1_earlier.to_numpy(),
        }),
        pd.DataFrame({
            "period": "1986–2015",
            "annual_temperature_c": h1_later.to_numpy(),
        }),
    ],
    ignore_index=True,
)

h1_figure = px.box(
    h1_plot_data,
    x="period",
    y="annual_temperature_c",
    color="period",
    points="all",
    color_discrete_map={
        "1956–1985": "#4C78A8",
        "1986–2015": "#E45756",
    },
    title="Annual Global Temperature by Comparison Period",
    labels={
        "period": "Period",
        "annual_temperature_c": "Annual temperature (°C)",
    },
    template="plotly_white",
)

h1_figure.update_layout(showlegend=False)
h1_figure.show()

### Hypothesis 1 Conclusion

Hypothesis 1 is **supported**.

The mean global land-and-ocean temperature was approximately **15.318 °C** during 1956–1985 and **15.701 °C** during 1986–2015. The later period was therefore approximately **0.383 °C warmer**.

The Welch test produced a p-value below 0.05, and Cohen's d indicated a large difference between the periods.

This result describes an association with time in the historical dataset. It does not independently identify or prove the physical causes of warming.

## Hypothesis 2

> Average global land-temperature measurement uncertainty is higher before 1900 than after 1950.

### Method

- Complete annual averages are used.
- The early group contains complete years before 1900.
- The later group contains complete years from 1950 onward.
- A two-sided Welch t-test and Cohen's d are reported.

In [7]:
global_land_annual = global_annual.loc[
    global_annual["land_months"].eq(12)
].copy()

h2_early = global_land_annual.loc[
    global_land_annual["year"] < 1900,
    "land_average_uncertainty_c",
]

h2_recent = global_land_annual.loc[
    global_land_annual["year"] >= 1950,
    "land_average_uncertainty_c",
]

h2_test = stats.ttest_ind(
    h2_early,
    h2_recent,
    equal_var=False,
)

h2_difference_c = h2_early.mean() - h2_recent.mean()
h2_effect_size = cohens_d(h2_early, h2_recent)

print(f"Before 1900 observations: {len(h2_early)}")
print(f"From 1950 observations: {len(h2_recent)}")
print(f"Before 1900 mean: {h2_early.mean():.4f} °C")
print(f"From 1950 mean: {h2_recent.mean():.4f} °C")
print(f"Difference: {h2_difference_c:.4f} °C")
print(f"Welch t-statistic: {h2_test.statistic:.4f}")
print(f"Two-sided p-value: {h2_test.pvalue:.3e}")
print(f"Cohen's d: {h2_effect_size:.3f}")

Before 1900 observations: 147
From 1950 observations: 66
Before 1900 mean: 1.5237 °C
From 1950 mean: 0.1002 °C
Difference: 1.4234 °C
Welch t-statistic: 18.1935
Two-sided p-value: 1.918e-39
Cohen's d: 1.806


In [8]:
h2_line_figure = px.line(
    global_land_annual,
    x="year",
    y="land_average_uncertainty_c",
    title="Reported Global Land-Temperature Uncertainty",
    labels={
        "year": "Year",
        "land_average_uncertainty_c":
            "Average reported uncertainty (°C)",
    },
    template="plotly_white",
)

h2_line_figure.update_traces(
    line_color="#6F4E7C",
    hovertemplate=(
        "Year: %{x}<br>"
        "Average uncertainty: %{y:.3f} °C"
        "<extra></extra>"
    ),
)

h2_line_figure.add_vrect(
    x0=1753,
    x1=1899,
    fillcolor="#F2CF5B",
    opacity=0.15,
    line_width=0,
    annotation_text="Before 1900",
)

h2_line_figure.add_vrect(
    x0=1950,
    x1=2015,
    fillcolor="#59A14F",
    opacity=0.12,
    line_width=0,
    annotation_text="1950 onward",
)

h2_line_figure.show()

In [9]:
h2_plot_data = pd.concat(
    [
        pd.DataFrame({
            "period": "Before 1900",
            "annual_uncertainty_c": h2_early.to_numpy(),
        }),
        pd.DataFrame({
            "period": "1950 onward",
            "annual_uncertainty_c": h2_recent.to_numpy(),
        }),
    ],
    ignore_index=True,
)

h2_box_figure = px.box(
    h2_plot_data,
    x="period",
    y="annual_uncertainty_c",
    color="period",
    points="outliers",
    color_discrete_map={
        "Before 1900": "#F2CF5B",
        "1950 onward": "#59A14F",
    },
    title="Measurement Uncertainty by Historical Period",
    labels={
        "period": "Period",
        "annual_uncertainty_c":
            "Average reported uncertainty (°C)",
    },
    template="plotly_white",
)

h2_box_figure.update_layout(showlegend=False)
h2_box_figure.show()

### Hypothesis 2 Conclusion

Hypothesis 2 is **supported**.

Average reported uncertainty was approximately **1.524 °C** before 1900 and **0.100 °C** from 1950 onward. Early-period uncertainty was therefore about **1.423 °C higher**.

The result is consistent with improvements in measurement coverage, methods, and data availability. However, this dataset alone cannot determine which specific historical improvements caused the reduction.

The two periods contain different numbers of years, and neighbouring annual observations may be correlated. The very small p-value should therefore be interpreted alongside the visual pattern and effect size rather than as the only evidence.

## Country Temperature Anomalies

Comparing absolute average temperatures between countries can be misleading because countries have different climates.

Each country's annual temperature is therefore compared with its own 1951–1980 baseline. Only complete years containing 12 monthly observations are retained.

In [10]:
country_annual_all = (
    country_clean
    .groupby(["country", "year"], as_index=False)
    .agg(
        average_temperature_c=(
            "average_temperature_c",
            "mean",
        ),
        average_uncertainty_c=(
            "average_temperature_uncertainty_c",
            "mean",
        ),
        months_observed=(
            "average_temperature_c",
            "count",
        ),
    )
)

country_annual = country_annual_all.loc[
    country_annual_all["months_observed"].eq(12)
].copy()

country_baselines = (
    country_annual.loc[
        country_annual["year"].between(1951, 1980)
    ]
    .groupby("country", as_index=False)
    .agg(
        baseline_temperature_c=(
            "average_temperature_c",
            "mean",
        ),
        baseline_years=(
            "year",
            "nunique",
        ),
    )
)

country_annual = country_annual.merge(
    country_baselines,
    on="country",
    how="left",
    validate="many_to_one",
)

country_annual["temperature_anomaly_c"] = (
    country_annual["average_temperature_c"]
    - country_annual["baseline_temperature_c"]
)

country_annual = (
    country_annual
    .sort_values(["country", "year"])
    .reset_index(drop=True)
)

assert country_annual["baseline_years"].eq(30).all()
assert country_annual["temperature_anomaly_c"].notna().all()

print(f"Country annual shape: {country_annual.shape}")
print(
    "Country labels:",
    country_annual["country"].nunique(),
)
print(
    "Coverage:",
    country_annual["year"].min(),
    "to",
    country_annual["year"].max(),
)

Country annual shape: (44330, 8)
Country labels: 242
Coverage: 1753 to 2012


In [11]:
selected_countries = [
    "Sweden",
    "United Kingdom",
    "United States",
    "India",
    "Brazil",
    "Australia",
]

country_plot_data = country_annual.loc[
    country_annual["country"].isin(selected_countries)
    & country_annual["year"].ge(1900)
].copy()

country_plot_data["anomaly_10_year_mean_c"] = (
    country_plot_data
    .groupby("country")["temperature_anomaly_c"]
    .transform(
        lambda values: values.rolling(
            window=10,
            min_periods=10,
        ).mean()
    )
)

country_figure = px.line(
    country_plot_data,
    x="year",
    y="anomaly_10_year_mean_c",
    color="country",
    title=(
        "Selected Country Temperature Anomalies: "
        "10-Year Rolling Mean"
    ),
    labels={
        "year": "Year",
        "anomaly_10_year_mean_c":
            "Temperature anomaly (°C)",
        "country": "Country",
    },
    template="plotly_white",
)

country_figure.add_hline(
    y=0,
    line_dash="dash",
    line_color="#555555",
)

country_figure.update_layout(
    hovermode="x unified",
    legend_title="Country",
)

country_figure.show()

### Country Comparison Interpretation

The selected countries have different absolute climates, so their country-specific anomalies are compared instead of their absolute temperatures.

The rolling averages reduce short-term variation and reveal longer-term patterns. The chart is intended for exploration rather than for ranking countries as more or less responsible for climate change.

In [12]:
hypothesis_results = pd.DataFrame([
    {
        "hypothesis": "H1",
        "comparison": "1986–2015 minus 1956–1985",
        "group_1_mean_c": h1_earlier.mean(),
        "group_2_mean_c": h1_later.mean(),
        "directional_difference_c": h1_difference_c,
        "welch_t_statistic": h1_test.statistic,
        "two_sided_p_value": h1_test.pvalue,
        "cohens_d": h1_effect_size,
        "conclusion": "Supported",
    },
    {
        "hypothesis": "H2",
        "comparison": "Before 1900 minus 1950 onward",
        "group_1_mean_c": h2_early.mean(),
        "group_2_mean_c": h2_recent.mean(),
        "directional_difference_c": h2_difference_c,
        "welch_t_statistic": h2_test.statistic,
        "two_sided_p_value": h2_test.pvalue,
        "cohens_d": h2_effect_size,
        "conclusion": "Supported",
    },
])

display(hypothesis_results)

,hypothesis,comparison,group_1_mean_c,group_2_mean_c,directional_difference_c,welch_t_statistic,two_sided_p_value,cohens_d,conclusion
0,H1,1986–2015 minus 1956–1985,15.318311,15.701442,0.383131,10.500891,3.207412e-14,2.711318,Supported
1,H2,Before 1900 minus 1950 onward,1.523695,0.100246,1.423449,18.193477,1.917640e-39,1.805786,Supported


In [13]:
GLOBAL_ANNUAL_OUTPUT = (
    PROCESSED_FOLDER / "global_annual_summary.csv"
)

COUNTRY_ANNUAL_OUTPUT = (
    PROCESSED_FOLDER / "country_annual_summary.csv"
)

HYPOTHESIS_OUTPUT = (
    PROCESSED_FOLDER / "hypothesis_results.csv"
)

global_annual.to_csv(
    GLOBAL_ANNUAL_OUTPUT,
    index=False,
)

country_annual.to_csv(
    COUNTRY_ANNUAL_OUTPUT,
    index=False,
)

hypothesis_results.to_csv(
    HYPOTHESIS_OUTPUT,
    index=False,
)

# Reload the files to verify that they were written correctly.
global_annual_check = pd.read_csv(GLOBAL_ANNUAL_OUTPUT)
country_annual_check = pd.read_csv(COUNTRY_ANNUAL_OUTPUT)
hypothesis_check = pd.read_csv(HYPOTHESIS_OUTPUT)

assert global_annual_check.shape == global_annual.shape
assert country_annual_check.shape == country_annual.shape
assert hypothesis_check.shape == hypothesis_results.shape

print("All analytical datasets were exported successfully.")
print(f"Global annual: {global_annual_check.shape}")
print(f"Country annual: {country_annual_check.shape}")
print(f"Hypotheses: {hypothesis_check.shape}")

All analytical datasets were exported successfully.
Global annual: (266, 8)
Country annual: (44330, 8)
Hypotheses: (2, 9)


## Exploratory Analysis Conclusions

- The global land-and-ocean annual series contains 166 complete years from 1850 through 2015.
- The temperature anomaly trend shows increasingly warm conditions relative to the 1951–1980 project baseline.
- Hypothesis 1 was supported: 1986–2015 was approximately 0.383 °C warmer than 1956–1985.
- Hypothesis 2 was supported: reported uncertainty before 1900 was approximately 1.423 °C higher than uncertainty from 1950 onward.
- Both comparisons produced p-values below 0.05 and large effect sizes.
- Annual aggregation reduces monthly pseudoreplication but does not entirely remove time dependence between observations.
- Country-specific anomalies allow countries with different climates to be compared more responsibly than absolute temperatures.
- Only complete country years were retained, so the incomplete year 2013 was excluded.
- The visual and statistical findings describe this historical dataset and must not be presented as proof of climate causation or as a current climate forecast.